# BERT Fine-Tuning for Sentiment Classification

**Run this notebook on Google Colab** (Runtime → Change runtime type → T4 GPU)

This notebook fine-tunes `deepset/gbert-base` on the German Sentiment dataset
and evaluates it against our Classical ML baselines.

Outputs:
- Trained model saved to Google Drive (or downloaded)
- Metrics JSON for comparison with other models

In [ ]:
# Install dependencies (Colab already has torch + transformers, but ensure versions)
!pip install -q datasets accelerate scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 1. Load Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("sepidmnorozy/German_sentiment")

LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_LABELS = 3

print(f"Train: {len(ds['train'])} samples")
print(f"Val:   {len(ds['validation'])} samples")
print(f"Test:  {len(ds['test'])} samples")
print(f"\nLabel distribution (train):")
print(pd.Series(ds['train']['label']).value_counts().sort_index())

## 2. Tokenize

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "deepset/gbert-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )

tokenized = ds.map(tokenize_fn, batched=True, batch_size=256)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized. Example length: {len(tokenized['train'][0]['input_ids'])}")

## 3. Fine-Tune gbert-base

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1_weighted": float(f1_score(labels, preds, average="weighted")),
        "f1_macro": float(f1_score(labels, preds, average="macro")),
        "precision_weighted": float(precision_score(labels, preds, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(labels, preds, average="weighted", zero_division=0)),
    }

training_args = TrainingArguments(
    output_dir="./bert_checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}...")
start = time.perf_counter()
train_result = trainer.train()
duration = time.perf_counter() - start
print(f"\nTraining complete in {duration:.1f}s")
print(f"Final train loss: {train_result.training_loss:.4f}")

## 4. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict on test set
test_output = trainer.predict(tokenized["test"])
test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_logits, axis=-1)
test_labels = np.array(ds["test"]["label"])

# Metrics
test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_preds)),
    "f1_weighted": float(f1_score(test_labels, test_preds, average="weighted")),
    "f1_macro": float(f1_score(test_labels, test_preds, average="macro")),
    "precision_weighted": float(precision_score(test_labels, test_preds, average="weighted", zero_division=0)),
    "recall_weighted": float(recall_score(test_labels, test_preds, average="weighted", zero_division=0)),
}

# ROC-AUC
try:
    test_metrics["roc_auc_weighted"] = float(
        roc_auc_score(test_labels, test_probs, multi_class="ovr", average="weighted")
    )
except ValueError:
    test_metrics["roc_auc_weighted"] = None

print("=" * 60)
print("TEST RESULTS: gbert-base fine-tuned")
print("=" * 60)
for k, v in test_metrics.items():
    if v is not None:
        print(f"  {k:25s}: {v:.4f}")

print(f"\nClassification Report:")
print(classification_report(
    test_labels, test_preds, target_names=LABEL_NAMES,
    labels=list(range(NUM_LABELS)), zero_division=0
))

print(f"Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

## 5. Measure Inference Latency

In [ ]:
# Latency measurement on 100 samples
sample_texts = ds["test"]["text"][:100]

model.eval()
times = []
for _ in range(3):
    start = time.perf_counter()
    encoded = tokenizer(
        sample_texts, max_length=MAX_LENGTH, padding=True,
        truncation=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        _ = model(**encoded)
    times.append(time.perf_counter() - start)

avg_time = np.mean(times)
latency = {
    "total_ms": round(avg_time * 1000, 2),
    "per_sample_ms": round((avg_time / 100) * 1000, 2),
    "samples_per_second": round(100 / avg_time, 1),
}

print(f"Latency (100 samples, avg of 3 runs):")
for k, v in latency.items():
    print(f"  {k}: {v}")

## 6. Save Results

In [ ]:
# Save metrics JSON (download this file)
all_results = {
    "model": "bert_finetuned_gbert_base",
    "base_model": MODEL_NAME,
    "dataset": "sepidmnorozy/German_sentiment",
    "train": {
        "train_duration_s": round(duration, 2),
        "train_loss": round(train_result.training_loss, 4),
    },
    "test": test_metrics,
    "latency": latency,
    "classification_report": classification_report(
        test_labels, test_preds, target_names=LABEL_NAMES,
        labels=list(range(NUM_LABELS)), output_dict=True, zero_division=0
    ),
    "confusion_matrix": confusion_matrix(test_labels, test_preds).tolist(),
}

with open("bert_finetuned_metrics.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Saved metrics to bert_finetuned_metrics.json")

In [ ]:
# Save model (to Google Drive or download)
SAVE_TO_DRIVE = True  # Set False to download instead

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    save_path = "/content/drive/MyDrive/models/bert_german_sentiment"
else:
    save_path = "./bert_german_sentiment"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Also save metrics alongside model
import shutil
shutil.copy("bert_finetuned_metrics.json", f"{save_path}/metrics.json")
print("Metrics copied to model directory")

In [ ]:
# Download metrics file (click the download link in output)
try:
    from google.colab import files
    files.download("bert_finetuned_metrics.json")
except ImportError:
    print("Not in Colab - file saved locally.")

## 7. Quick Comparison with Classical ML

Paste the classical ML results here for comparison:

| Model | F1 (weighted) | Accuracy | Latency (ms/sample) |
|-------|--------------|----------|--------------------|
| Naive Bayes | 0.7956 | 0.8584 | 0.09 |
| Logistic Regression | 0.8505 | 0.8503 | 0.07 |
| SVM | 0.8562 | 0.8725 | 0.11 |
| **gbert-base (this notebook)** | **see above** | **see above** | **see above** |